# Reranker prompt token-length stats

Takes one KernelBench run (an `eval_results.json` + staged `level_{L}_problem_{P}_sample_{S}_kernel.py`
files) and, for every candidate kernel, reconstructs **exactly** the cross-encoder prompt the reranker
feeds to the model:

```
INSTRUCTION + reference_arch + SEPARATOR + candidate_kernel + EOS
```

(format is the single source of truth in `reranker/src/encoding.py`). Each prompt is run through the
**Qwen reranker tokenizer** and we report the distribution of the **full, untruncated** prompt length —
i.e. how many tokens the prompt *would* need before `max_length`/`reserve_ref_tokens` truncation kicks in.
That is the number that tells you how to size `max_length`.

Reported per component (instruction, reference, kernel, total): **mean, median, p90, p95, p99, max**.

In [ ]:
# --- Parameters ---------------------------------------------------------
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# The run that solves the level (must contain eval_results.json + staged kernels).
RUN_DIR = REPO_ROOT / "runs" / "Qwen3-Coder-30B-A3B-Instruct_level1_triton"
LEVEL = 1

# KernelBench dataset root (the dir that contains level1/, level2/, ...).
KERNELBENCH_DIR = REPO_ROOT / "KernelBench"

# Tokenizer + truncation budget (mirrors reranker/configs/*.yaml -> model.*).
BASE_MODEL = "Qwen/Qwen3-Reranker-0.6B"
MAX_LENGTH = 4096          # for reporting how many prompts overflow
RESERVE_REF_TOKENS = 1024  # informational only

print("repo root:", REPO_ROOT)
print("run dir  :", RUN_DIR)
assert (RUN_DIR / "eval_results.json").is_file(), f"no eval_results.json in {RUN_DIR}"

In [ ]:
# --- Load the prompt format + tokenizer + KernelBench reference sources --
import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# The exact INSTRUCTION / SEPARATOR strings the reranker uses (single source of truth).
from reranker.src.encoding import INSTRUCTION, SEPARATOR
from kernelbench.dataset import construct_kernelbench_dataset, fetch_ref_arch_from_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print("eos_token_id:", tokenizer.eos_token_id)

kb_dataset = construct_kernelbench_dataset(level=LEVEL, source="local", base_path=str(KERNELBENCH_DIR))

# Precompute the static instruction / separator token ids (no special tokens), like SequenceEncoder.
instr_ids = tokenizer.encode(INSTRUCTION, add_special_tokens=False)
sep_ids = tokenizer.encode(SEPARATOR, add_special_tokens=False)
n_special = 1 if tokenizer.eos_token_id is not None else 0  # trailing EOS
static_overhead = len(instr_ids) + len(sep_ids) + n_special
print(f"instruction tokens: {len(instr_ids)} | separator tokens: {len(sep_ids)} | EOS: {n_special}")
print(f"static overhead per prompt: {static_overhead} tokens")

In [ ]:
# --- Walk the run: tokenize every (reference, candidate kernel) prompt ----
import json

eval_results = json.loads((RUN_DIR / "eval_results.json").read_text())

rows = []          # one dict per candidate kernel
missing_kernel = 0
ref_tok_cache = {}  # problem_id -> reference token length (ref is shared across samples)

for problem_id_str, samples in eval_results.items():
    problem_id = int(problem_id_str)
    try:
        _, problem_name, ref_arch_src = fetch_ref_arch_from_dataset(kb_dataset, problem_id)
    except ValueError:
        print(f"[WARN] problem {problem_id} not in KernelBench level {LEVEL}")
        continue

    if problem_id not in ref_tok_cache:
        ref_tok_cache[problem_id] = len(tokenizer.encode(ref_arch_src, add_special_tokens=False))
    n_ref = ref_tok_cache[problem_id]

    for sample in samples:
        sample_id = int(sample["sample_id"])
        kpath = RUN_DIR / f"level_{LEVEL}_problem_{problem_id}_sample_{sample_id}_kernel.py"
        if not kpath.is_file():
            missing_kernel += 1
            continue
        kernel_src = kpath.read_text()
        n_kernel = len(tokenizer.encode(kernel_src, add_special_tokens=False))
        n_total = static_overhead + n_ref + n_kernel  # full, untruncated prompt length
        rows.append({
            "problem_id": problem_id,
            "problem_name": problem_name,
            "sample_id": sample_id,
            "compiled": bool(sample.get("compiled", False)),
            "correct": bool(sample.get("correctness", False)),
            "ref_tokens": n_ref,
            "kernel_tokens": n_kernel,
            "total_tokens": n_total,
        })

print(f"prompts tokenized : {len(rows)}")
print(f"missing kernels   : {missing_kernel} (eval entry but no staged .py)")
assert rows, "no prompts produced — check RUN_DIR / LEVEL"

In [ ]:
# --- Stats: mean / median / p90 / p95 / p99 / max ------------------------
import numpy as np
import pandas as pd

df = pd.DataFrame(rows)

def summarize(values):
    v = np.asarray(values, dtype=float)
    return {
        "mean": v.mean(),
        "median": np.median(v),
        "p90": np.percentile(v, 90),
        "p95": np.percentile(v, 95),
        "p99": np.percentile(v, 99),
        "max": v.max(),
    }

stats = pd.DataFrame({
    "reference": summarize(df["ref_tokens"]),
    "kernel": summarize(df["kernel_tokens"]),
    "total_prompt": summarize(df["total_tokens"]),
}).T.round(1)
stats = stats[["mean", "median", "p90", "p95", "p99", "max"]]
print(f"Run: {RUN_DIR.name}  |  level {LEVEL}  |  {len(df)} candidate prompts\n")
stats

In [ ]:
# --- How many prompts overflow the configured max_length? ----------------
over = int((df["total_tokens"] > MAX_LENGTH).sum())
print(f"max_length = {MAX_LENGTH} (reserve_ref_tokens = {RESERVE_REF_TOKENS})")
print(f"prompts exceeding max_length: {over} / {len(df)}  ({100 * over / len(df):.1f}%)")
print(f"longest full prompt         : {int(df['total_tokens'].max())} tokens")
print(f"-> {'all prompts fit' if over == 0 else 'some prompts are truncated'} at max_length={MAX_LENGTH}")

In [ ]:
# --- (optional) histogram of total prompt length -------------------------
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["total_tokens"], bins=50, color="#4c72b0", edgecolor="white")
for q, c in [(90, "#dd8452"), (95, "#c44e52"), (99, "#8172b3")]:
    x = np.percentile(df["total_tokens"], q)
    ax.axvline(x, color=c, ls="--", lw=1, label=f"p{q} = {x:.0f}")
ax.axvline(MAX_LENGTH, color="k", ls=":", lw=1.5, label=f"max_length = {MAX_LENGTH}")
ax.set_xlabel("full (untruncated) prompt length [tokens]")
ax.set_ylabel("# candidate prompts")
ax.set_title(f"{RUN_DIR.name}  (level {LEVEL})")
ax.legend()
plt.tight_layout()
plt.show()